In [10]:
# Import libraries
import pandas as pd


In [11]:
df=pd.read_csv(r"D:\Olist\ml_dataset_labeled.csv")
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,max_item_price,min_item_price,total_freight,total_items,seller_id,total_pyment_value,max_installments,primary_pyment_type,avg_review_score,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,29.99,29.99,8.72,1.0,3504c0cb71d7fa48d967e0e4c94d59d9,38.71,1.0,voucher,4.0,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,118.70,118.70,22.76,1.0,289cdb325fb7e7f891c38608bf9e0962,141.46,1.0,boleto,4.0,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,159.90,159.90,19.22,1.0,4869f7a5dfa277a7dca6462dcf3b52b2,179.12,3.0,credit_card,5.0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,45.00,45.00,27.20,1.0,66922902710d126a0e7d26b0e3805106,72.20,1.0,credit_card,5.0,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,19.90,19.90,8.72,1.0,2c9e548be18521d1c43cde1c582c6de8,28.62,1.0,credit_card,5.0,0


### Split the data before any deep analysis Using (Time Based Split)
- Prevents Data Leakage: In real-world supply chain and e-commerce scenarios, predicting order delays requires relying strictly on past historical data. A random split would mix future orders into the training set, effectively letting the model "look into the future."
- Simulates Real-World Deployment: In production, machine learning models predict future outcomes using past records. Splitting chronologically mirrors how the model will perform once deployed.
- Captures Temporal Drift: External conditions such as logistics routes, carrier efficiency, seasonal peaks (e.g., Black Friday), and order volumes evolve over time. A time-based evaluation tests how well the model generalizes to shifting operational dynamics.


In [12]:
# Convert timestamp to datetime and sort it
df["order_purchase_timestamp"]=pd.to_datetime(df["order_purchase_timestamp"])
df=df.sort_values("order_purchase_timestamp").reset_index(drop=True)

In [13]:
# Split by position to avoid leakage
n=len(df)
train_end=int(n*0.80)
val_end=int(n*0.90)

In [14]:
# Create train, validation, and test dataframes
train_df=df.iloc[:train_end].copy()
val_df=df.iloc[train_end:val_end].copy()
test_df=df.iloc[val_end:].copy()

In [15]:
# Print shapes and time periods
print(f"Train shape:      {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape:       {test_df.shape}")
print("=="*50)
print(f"Train period:      {train_df['order_purchase_timestamp'].iloc[0]} -> {train_df['order_purchase_timestamp'].iloc[-1]}")
print(f"validation period: {val_df['order_purchase_timestamp'].iloc[0]} ->   {val_df['order_purchase_timestamp'].iloc[-1]}")
print(f"Test period:       {test_df['order_purchase_timestamp'].iloc[0]} ->  {test_df['order_purchase_timestamp'].iloc[-1]}")

Train shape:      (79552, 24)
Validation shape: (9944, 24)
Test shape:       (9945, 24)
Train period:      2016-09-04 21:15:19 -> 2018-05-24 17:51:08
validation period: 2018-05-24 18:04:31 ->   2018-07-18 10:31:28
Test period:       2018-07-18 10:33:46 ->  2018-10-17 17:30:18


### # Separate the features and target

In [16]:
target_col="is_late"

# Train split
X_train=train_df.drop(columns=[target_col])
y_train=train_df[target_col]

# Validation split
X_val=val_df.drop(columns=[target_col])
y_val=val_df[target_col]

# Test split
X_test=test_df.drop(columns=[target_col])
y_test=test_df[target_col]

In [17]:
# Check the class balance in each split
print("Train target distribution:")
print(y_train.value_counts(normalize=True))

print("Validation target distribution:")
print(y_val.value_counts(normalize=True))

print("Test target distribution:")
print(y_test.value_counts(normalize=True))


Train target distribution:
is_late
0    0.914471
1    0.085529
Name: proportion, dtype: float64
Validation target distribution:
is_late
0    0.98029
1    0.01971
Name: proportion, dtype: float64
Test target distribution:
is_late
0    0.916843
1    0.083157
Name: proportion, dtype: float64


### Artifact: train, validation, and test files

In [18]:
train_df.to_csv("train.csv",index=False)
val_df.to_csv("validation.csv",index=False)
test_df.to_csv("test.csv",index=False)

print("Artifacts saved successfully: train.csv, validation.csv, and test.csv")

Artifacts saved successfully: train.csv, validation.csv, and test.csv
